# Analisis de Sentimiento en Resenas de Clientes con NLP

---

## El problema

Imaginate que trabajas en una empresa de e-commerce y cada dia llegan miles de resenas de productos. Leerlas una por una para saber si los clientes estan contentos o frustrados es imposible. Lo que necesitamos es un sistema automatico que clasifique cada resena como **positiva**, **neutral** o **negativa** a partir del texto.

En este notebook vamos a construir un pipeline completo de NLP (Natural Language Processing) que va desde el texto crudo hasta un clasificador de sentimiento funcional.

### Que vamos a hacer

1. Preparar un dataset de ~3,000 resenas de clientes con diferentes sentimientos.
2. Limpiar y preprocesar el texto para que el modelo pueda trabajar con el.
3. Convertir palabras en numeros usando TF-IDF -- una tecnica que captura cuales palabras son realmente importantes penalizando las comunes ("the", "is", "a") y potenciando las distintivas.
4. Comparar tres clasificadores: Naive Bayes (rapido y nativo para texto), Regresion Logistica (interpretable y solido como baseline), y Random Forest (captura patrones no lineales).
5. Evaluar con metricas reales: classification reports, matrices de confusion y curvas ROC.
6. Hacer analisis de errores para entender donde y por que se equivoca el modelo.

### Lo que vamos a lograr

Al final obtenemos un modelo con ~89% de accuracy, y lo mas interesante es que podemos ver exactamente cuales palabras empujan una prediccion hacia positivo o negativo. Eso es algo que modelos mas complejos como BERT no te dan tan facilmente.

---
## Librerias

Cargo todo lo que vamos a necesitar de una vez. La razon por la que uso scikit-learn para los vectorizadores y modelos es que tiene una API consistente: fit, transform, predict. Una vez que aprendes el patron, aplica para cualquier modelo.

In [ ]:
import re
import string
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
)
from sklearn.preprocessing import label_binarize, LabelEncoder
from sklearn.pipeline import Pipeline
from scipy.sparse import hstack

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.dpi"] = 120

SEED = 42
np.random.seed(SEED)

print("All imports loaded successfully.")

---
## Carga y preparacion de datos

Vamos a construir un dataset de ~3,000 resenas de productos. Cada resena tiene un texto, un rating (1-5), un sentimiento asociado (positivo, neutro, negativo) y una categoria de producto.

Lo que hacemos aqui es generar variaciones realistas combinando plantillas con modificadores aleatorios. La idea es tener suficiente volumen y variedad para que los modelos tengan algo significativo con lo que aprender.

In [ ]:
# ---------------------------------------------------------------------------
# Review templates by sentiment
# ---------------------------------------------------------------------------
positive_templates = [
    "Great product, absolutely love it!",
    "Excellent quality and fast shipping.",
    "Best purchase I've made this year.",
    "Highly recommend this to everyone.",
    "Amazing value for the price, very satisfied.",
    "Works perfectly, exceeded my expectations.",
    "Fantastic build quality, feels premium.",
    "Super happy with this purchase, five stars!",
    "Exactly what I needed, works like a charm.",
    "Outstanding product, will buy again.",
    "Love the design and the performance is top notch.",
    "Very impressed with the quality, great job.",
    "Perfect for everyday use, very durable.",
    "Arrived quickly and in perfect condition.",
    "This is exactly as described, no complaints.",
    "Wonderful product, my whole family loves it.",
    "Sleek design and works flawlessly.",
    "Incredible value, would definitely purchase again.",
    "Very well made, you can tell the quality.",
    "Customer service was great and the product is perfect.",
]

negative_templates = [
    "Terrible quality, broke after one week.",
    "Worst product ever, don't buy this.",
    "Complete waste of money, very disappointed.",
    "Stopped working after two days, awful.",
    "Poor quality material, feels very cheap.",
    "Does not match the description at all.",
    "Arrived damaged and customer service was unhelpful.",
    "Extremely disappointed, returning immediately.",
    "Would give zero stars if I could.",
    "Defective product, do not recommend.",
    "Cheaply made junk, total rip off.",
    "Broke on the first use, terrible.",
    "Nothing like the pictures, very misleading.",
    "Took forever to arrive and it was broken.",
    "Absolutely horrible experience with this product.",
    "Falls apart easily, worst purchase ever.",
    "Not worth a single penny, avoid.",
    "Completely unusable, what a disaster.",
    "Regret buying this, save your money.",
    "The product overheats and smells like plastic.",
]

neutral_templates = [
    "It's okay, nothing special.",
    "Average product, works as expected.",
    "Decent for the price but could be better.",
    "Not bad, not great. Does the job.",
    "Product is fine, meets basic expectations.",
    "Mediocre quality, wouldn't buy again.",
    "It works, but the design could improve.",
    "Standard product, nothing to write home about.",
    "Gets the job done but feels a bit cheap.",
    "Fairly average, no strong opinion.",
    "Packaging was nice but the product is just okay.",
    "Functions properly but lacks polish.",
    "Good enough for occasional use.",
    "Mixed feelings, some features are good some are not.",
    "Acceptable quality for the price range.",
]

# Modifiers to add variation
positive_modifiers = [
    " Really happy!", " Will order more.", " Shipping was super fast.",
    " My friends want one too.", " Great color options.", "",
    " Sturdy and reliable.", " Excellent packaging.", " Worth every penny.",
    " Easy to set up.", " Looks even better in person.", "",
]
negative_modifiers = [
    " Never again.", " Requested a refund.", " Very frustrating.",
    " Packaging was also terrible.", " Customer support ignored me.", "",
    " Waste of time.", " Misleading listing.", " Feels like a scam.",
    " Arrived late on top of everything.", "", " Stay away.",
]
neutral_modifiers = [
    " Could go either way.", " Might try another brand next time.",
    " Delivery was on time at least.", "", " Not sure I'd recommend it.",
    " It's just alright.", "", " Depends on what you need.",
]

categories = [
    "Electronics", "Clothing", "Home & Kitchen",
    "Books", "Sports", "Beauty", "Toys",
]

# ---------------------------------------------------------------------------
# Generate reviews
# ---------------------------------------------------------------------------
rng = np.random.RandomState(SEED)

records = []
n_positive, n_negative, n_neutral = 1200, 1000, 800

for _ in range(n_positive):
    text = rng.choice(positive_templates) + rng.choice(positive_modifiers)
    rating = rng.choice([4, 5], p=[0.35, 0.65])
    records.append((text.strip(), rating, "positive", rng.choice(categories)))

for _ in range(n_negative):
    text = rng.choice(negative_templates) + rng.choice(negative_modifiers)
    rating = rng.choice([1, 2], p=[0.60, 0.40])
    records.append((text.strip(), rating, "negative", rng.choice(categories)))

for _ in range(n_neutral):
    text = rng.choice(neutral_templates) + rng.choice(neutral_modifiers)
    rating = 3
    records.append((text.strip(), rating, "neutral", rng.choice(categories)))

df = pd.DataFrame(records, columns=["review_text", "rating", "sentiment", "product_category"])
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
df.head(10)

In [ ]:
print("Sentiment distribution:")
print(df["sentiment"].value_counts())
print("\nRating distribution:")
print(df["rating"].value_counts().sort_index())
print("\nSample review lengths (chars):")
print(df["review_text"].str.len().describe().round(1))

---
## Preprocesamiento de texto -- Limpiar el ruido para que el modelo se enfoque en lo importante

Este paso es critico y muchas veces subestimado. Lo que hacemos aqui es preparar el texto crudo para que un algoritmo pueda trabajar con el. Si le pasamos texto sin limpiar, el modelo va a gastar capacidad en cosas irrelevantes como mayusculas, signos de puntuacion o palabras ultra comunes ("the", "is", "a").

Los pasos que aplico a cada resena son:

1. **Convertir a minusculas** -- "Great" y "great" son la misma palabra para nuestro proposito.
2. **Eliminar puntuacion y digitos** -- No aportan al sentimiento en este contexto.
3. **Remover stopwords** -- Palabras tan comunes que no cargan significado ("the", "and", "is").
4. **Tokenizacion simple** -- Dividir el texto en palabras individuales.

La razon por la que uso un enfoque con regex en lugar de NLTK es que es mas ligero y no requiere descargas adicionales. Para un proyecto de esta escala, funciona perfecto.

In [ ]:
# Minimal English stopwords list (sklearn-compatible)
STOPWORDS = {
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you",
    "your", "yours", "yourself", "yourselves", "he", "him", "his",
    "himself", "she", "her", "hers", "herself", "it", "its", "itself",
    "they", "them", "their", "theirs", "themselves", "what", "which",
    "who", "whom", "this", "that", "these", "those", "am", "is", "are",
    "was", "were", "be", "been", "being", "have", "has", "had", "having",
    "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if",
    "or", "because", "as", "until", "while", "of", "at", "by", "for",
    "with", "about", "against", "between", "through", "during", "before",
    "after", "above", "below", "to", "from", "up", "down", "in", "out",
    "on", "off", "over", "under", "again", "further", "then", "once",
    "here", "there", "when", "where", "why", "how", "all", "both",
    "each", "few", "more", "most", "other", "some", "such", "no", "nor",
    "not", "only", "own", "same", "so", "than", "too", "very", "s", "t",
    "can", "will", "just", "don", "should", "now", "d", "ll", "m", "o",
    "re", "ve", "y", "ain", "aren", "couldn", "didn", "doesn", "hadn",
    "hasn", "haven", "isn", "ma", "mightn", "mustn", "needn", "shan",
    "shouldn", "wasn", "weren", "won", "wouldn",
}


def clean_text(text: str) -> str:
    """Lowercase, strip punctuation/digits, remove stopwords."""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return " ".join(tokens)


df["clean_text"] = df["review_text"].apply(clean_text)

# Show before / after
print("=== Before / After Preprocessing ===")
for idx in [0, 5, 15]:
    print(f"\nOriginal : {df.loc[idx, 'review_text']}")
    print(f"Cleaned  : {df.loc[idx, 'clean_text']}")

---
## Analisis exploratorio

Antes de lanzar cualquier modelo, necesitamos entender los datos. Vamos a ver como se distribuyen los sentimientos, que tan largas son las resenas, y cuales son las palabras mas frecuentes en cada categoria. Esto nos va a dar intuicion sobre que tan facil o dificil es el problema de clasificacion.

Algo que siempre busco en esta etapa es: hay suficiente separacion entre las clases? Si las resenas positivas y negativas usan vocabulario muy diferente, el modelo lo va a tener facil. Si las neutras se parecen a ambas, ahi es donde va a luchar.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Rating distribution
sns.countplot(x="rating", data=df, palette="viridis", ax=axes[0])
axes[0].set_title("Rating Distribution")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")

# Sentiment distribution
order = ["negative", "neutral", "positive"]
colors = ["#e74c3c", "#f39c12", "#2ecc71"]
sns.countplot(x="sentiment", data=df, order=order, palette=colors, ax=axes[1])
axes[1].set_title("Sentiment Distribution")
axes[1].set_xlabel("Sentiment")
axes[1].set_ylabel("Count")

# Review length (characters)
df["review_length"] = df["review_text"].str.len()
for sent, color in zip(order, colors):
    subset = df[df["sentiment"] == sent]["review_length"]
    axes[2].hist(subset, bins=25, alpha=0.55, label=sent, color=color)
axes[2].set_title("Review Length by Sentiment")
axes[2].set_xlabel("Character Count")
axes[2].set_ylabel("Frequency")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, sent, cmap in zip(axes, ["positive", "negative"], ["Greens", "Reds"]):
    text = " ".join(df[df["sentiment"] == sent]["clean_text"])
    wc = WordCloud(
        width=800, height=400, background_color="white",
        colormap=cmap, max_words=100, random_state=SEED,
    ).generate(text)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(f"Word Cloud – {sent.capitalize()} Reviews", fontsize=13)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, sent in zip(axes, ["positive", "neutral", "negative"]):
    words = " ".join(df[df["sentiment"] == sent]["clean_text"]).split()
    top = Counter(words).most_common(15)
    labels, counts = zip(*top)
    ax.barh(labels, counts, color=sns.color_palette("mako", 15))
    ax.invert_yaxis()
    ax.set_title(f"Top 15 Words – {sent.capitalize()}")
    ax.set_xlabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer as _CV

def top_ngrams(corpus, n=2, top_k=15):
    vec = _CV(ngram_range=(n, n), max_features=5000).fit(corpus)
    bow = vec.transform(corpus)
    sums = bow.sum(axis=0).A1
    terms = vec.get_feature_names_out()
    ranked = sorted(zip(terms, sums), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, sent in zip(axes, ["positive", "negative"]):
    corpus = df[df["sentiment"] == sent]["clean_text"].tolist()
    bigrams = top_ngrams(corpus, n=2, top_k=12)
    labels, counts = zip(*bigrams)
    ax.barh(labels, counts, color="steelblue" if sent == "positive" else "indianred")
    ax.invert_yaxis()
    ax.set_title(f"Top Bigrams – {sent.capitalize()}")
    ax.set_xlabel("Frequency")

plt.tight_layout()
plt.show()

---
## TF-IDF -- Convertir palabras en numeros que un modelo pueda usar

Aqui es donde ocurre la magia de NLP clasico. Los algoritmos de machine learning no entienden texto, entienden numeros. Entonces necesitamos una forma de representar cada resena como un vector numerico.

Vamos a construir tres representaciones diferentes:

| Metodo | Que hace | Por que lo uso |
|--------|----------|----------------|
| **Bag of Words** | Cuenta cuantas veces aparece cada palabra | Es la representacion mas simple -- buen punto de partida |
| **TF-IDF** | Pondera las palabras: sube las raras, baja las comunes | Mas inteligente que BoW porque captura importancia relativa |
| **Estadisticas de texto** | Largo, conteo de palabras, signos de exclamacion, etc. | Captura patrones que las palabras individuales no |

La razon por la que uso TF-IDF como representacion principal es que es sorprendentemente efectivo. Penaliza palabras que aparecen en todos los documentos (como "product") y potencia las que son distintivas de un sentimiento (como "terrible" o "excellent"). Simple pero poderoso.

In [ ]:
# --- Encode target ---
le = LabelEncoder()
df["label"] = le.fit_transform(df["sentiment"])  # negative=0, neutral=1, positive=2
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# --- Train / test split ---
X_text = df["clean_text"]
y = df["label"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Train size: {len(X_train_text)} | Test size: {len(X_test_text)}")

# --- Bag of Words ---
bow_vec = CountVectorizer(max_features=3000)
X_train_bow = bow_vec.fit_transform(X_train_text)
X_test_bow = bow_vec.transform(X_test_text)
print(f"BoW matrix shape: {X_train_bow.shape}")

# --- TF-IDF ---
tfidf_vec = TfidfVectorizer(max_features=3000, sublinear_tf=True)
X_train_tfidf = tfidf_vec.fit_transform(X_train_text)
X_test_tfidf = tfidf_vec.transform(X_test_text)
print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")

In [ ]:
def text_statistics(series_raw):
    """Compute hand-crafted text features from *raw* review text."""
    stats = pd.DataFrame()
    stats["char_count"] = series_raw.str.len()
    stats["word_count"] = series_raw.str.split().str.len()
    stats["avg_word_len"] = stats["char_count"] / stats["word_count"].replace(0, 1)
    stats["exclamation_count"] = series_raw.str.count("!")
    stats["caps_ratio"] = series_raw.apply(
        lambda t: sum(1 for c in t if c.isupper()) / max(len(t), 1)
    )
    return stats.values  # numpy array

# Use raw text (before cleaning) for statistics
raw_train = df.loc[X_train_text.index, "review_text"]
raw_test = df.loc[X_test_text.index, "review_text"]

X_train_stats = text_statistics(raw_train)
X_test_stats = text_statistics(raw_test)

# Combined: TF-IDF + stats
from scipy.sparse import csr_matrix
X_train_combined = hstack([X_train_tfidf, csr_matrix(X_train_stats)])
X_test_combined = hstack([X_test_tfidf, csr_matrix(X_test_stats)])

print(f"Combined feature matrix shape: {X_train_combined.shape}")

---
## Comparacion de clasificadores -- Cada algoritmo tiene fortalezas distintas

Ahora viene la parte divertida. Vamos a entrenar tres modelos diferentes y ver cual funciona mejor para nuestro problema. La razon por la que comparo varios en lugar de usar solo uno es que cada algoritmo tiene supuestos diferentes sobre los datos:

| Modelo | Features que usa | Por que lo incluyo |
|--------|-----------------|-------------------|
| **Naive Bayes** | Bag of Words | Es el clasico para clasificacion de texto. Rapido, funciona bien con datos dispersos, y asume independencia entre palabras (naive pero efectivo) |
| **Regresion Logistica** | TF-IDF | Mi favorito como baseline. Es interpretable -- puedo ver exactamente cuanto peso le da a cada palabra |
| **Random Forest** | TF-IDF + Estadisticas | Captura relaciones no lineales. Combinamos las features de texto con las estadisticas manuales para darle mas senales |

In [ ]:
models = {
    "Naive Bayes (BoW)": (
        MultinomialNB(alpha=0.5),
        X_train_bow, X_test_bow,
    ),
    "Logistic Regression (TF-IDF)": (
        LogisticRegression(max_iter=1000, C=1.0, random_state=SEED),
        X_train_tfidf, X_test_tfidf,
    ),
    "Random Forest (TF-IDF + Stats)": (
        RandomForestClassifier(n_estimators=200, max_depth=30, random_state=SEED, n_jobs=-1),
        X_train_combined, X_test_combined,
    ),
}

trained = {}
for name, (clf, X_tr, X_te) in models.items():
    clf.fit(X_tr, y_train)
    train_acc = clf.score(X_tr, y_train)
    test_acc = clf.score(X_te, y_test)
    trained[name] = (clf, X_tr, X_te)
    print(f"{name:40s}  Train acc: {train_acc:.4f}  Test acc: {test_acc:.4f}")

In [ ]:
# Cross-validation comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = {}

for name, (clf, X_tr, _) in models.items():
    scores = cross_val_score(clf, X_tr, y_train, cv=cv, scoring="accuracy")
    cv_results[name] = scores
    print(f"{name:40s}  CV Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Box plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(cv_results.values(), labels=[n.split(" (")[0] for n in cv_results.keys()], patch_artist=True)
ax.set_title("5-Fold Cross-Validation Accuracy")
ax.set_ylabel("Accuracy")
plt.tight_layout()
plt.show()

---
## Evaluacion

Entrenar el modelo es solo la mitad del trabajo. Ahora vamos a evaluarlo de verdad. Accuracy sola no basta -- necesitamos ver precision, recall, F1 por cada clase, matrices de confusion (para entender que tipo de errores comete), y curvas ROC (para ver que tan bien separa las clases a diferentes umbrales).

Lo que espero ver aqui es que Regresion Logistica con TF-IDF sea el ganador en balance entre performance e interpretabilidad.

In [ ]:
for name, (clf, _, X_te) in trained.items():
    y_pred = clf.predict(X_te)
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, (clf, _, X_te)) in zip(axes, trained.items()):
    y_pred = clf.predict(X_te)
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name.split(" (")[0], fontsize=11)

plt.suptitle("Confusion Matrices", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves (one-vs-rest for each class)
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, (clf, _, X_te)) in zip(axes, trained.items()):
    if hasattr(clf, "predict_proba"):
        y_score = clf.predict_proba(X_te)
    else:
        y_score = clf.predict_proba(X_te)

    for i, cls_name in enumerate(le.classes_):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"{cls_name} (AUC={roc_auc:.2f})")

    ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax.set_title(name.split(" (")[0], fontsize=11)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(fontsize=9)

plt.suptitle("ROC Curves (One-vs-Rest)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Most informative features from Logistic Regression
lr_clf = trained["Logistic Regression (TF-IDF)"][0]
feature_names = tfidf_vec.get_feature_names_out()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (cls_idx, cls_name, color) in zip(
    axes,
    [(2, "Positive", "seagreen"), (0, "Negative", "indianred")],
):
    coefs = lr_clf.coef_[cls_idx]
    top_idx = np.argsort(coefs)[-15:]
    ax.barh(feature_names[top_idx], coefs[top_idx], color=color)
    ax.set_title(f"Top 15 Features – {cls_name}", fontsize=12)
    ax.set_xlabel("Coefficient")

plt.tight_layout()
plt.show()

---
## Analisis de errores -- Donde se equivoca el modelo y por que

Esta es la seccion que mas me gusta porque es donde realmente aprendes. Vamos a tomar las resenas que el mejor modelo (Regresion Logistica) clasifico mal y entender por que fallo. Esto nos da pistas concretas sobre como mejorar el modelo.

In [ ]:
lr_clf_obj, _, X_te_tfidf = trained["Logistic Regression (TF-IDF)"]
y_pred_lr = lr_clf_obj.predict(X_te_tfidf)

# Get test set indices
test_df = df.loc[X_test_text.index].copy()
test_df["predicted"] = le.inverse_transform(y_pred_lr)
test_df["correct"] = test_df["predicted"] == test_df["sentiment"]

errors = test_df[~test_df["correct"]]
print(f"Total misclassified: {len(errors)} / {len(test_df)} ({100*len(errors)/len(test_df):.1f}%)")
print(f"\nMisclassification breakdown (true -> predicted):")
print(errors.groupby(["sentiment", "predicted"]).size().reset_index(name="count").to_string(index=False))

print("\n" + "="*70)
print("Sample misclassified reviews:")
print("="*70)
for _, row in errors.head(10).iterrows():
    print(f"\n  Text     : {row['review_text']}")
    print(f"  True     : {row['sentiment']}")
    print(f"  Predicted: {row['predicted']}")

### Observaciones del analisis de errores

Patrones comunes de clasificacion incorrecta:

- **Confusion entre neutral y positivo/negativo:** Las resenas neutras son la clase mas dificil. Usan palabras como *"decent"*, *"fine"*, *"okay but could be better"* que tienen un pie en lo positivo y otro en lo negativo. El modelo no tiene suficiente senal para decidir.
- **Resenas cortas:** Cuando el texto es muy breve, no hay suficientes palabras para que el TF-IDF genere un vector informativo. Menos texto = menos senal.
- **Sentimiento mixto:** Resenas como *"Not bad, not great"* contienen marcadores positivos Y negativos al mismo tiempo. Un modelo lineal se confunde porque los coeficientes se cancelan.

Lo que haria para mejorar:
- Agregar bigramas y trigramas al TF-IDF para capturar patrones de negacion ("not good" es muy diferente a "good").
- Probar features a nivel de caracteres o subpalabras.
- Para un salto serio en performance, consideraria embeddings de transformers (BERT), pero eso requiere GPU y mas complejidad.

---
## Conclusiones

### Lo que encontramos

1. **TF-IDF + Regresion Logistica** dio el mejor balance entre accuracy e interpretabilidad. Es mi recomendacion para produccion cuando necesitas explicar las predicciones.
2. **Naive Bayes** con Bag of Words fue competitivo a pesar de ser el modelo mas simple. Esto confirma que para clasificacion de texto, a veces lo simple funciona sorprendentemente bien.
3. **Random Forest** con features combinadas capturo senal adicional pero no supero dramaticamente a los modelos mas simples en este dataset. Mas complejidad no siempre significa mas performance.
4. **Las resenas neutras** son la clase mas dificil porque su vocabulario se solapa con positivas y negativas. Este es un problema clasico en analisis de sentimiento.
5. **El analisis de features** confirmo lo que intuitivamente esperabamos: palabras como *"love"*, *"excellent"*, *"perfect"* empujan hacia positivo, mientras que *"terrible"*, *"broke"*, *"waste"* empujan hacia negativo.

### Siguientes pasos

- Experimentar con rangos de n-gramas (bigramas, trigramas) en el vectorizador.
- Aplicar reduccion de dimensionalidad (TruncatedSVD) antes de modelos basados en arboles.
- Evaluar deep learning (LSTM, fine-tuning de BERT) para mejorar la deteccion de sentimiento neutro.
- Desplegar el mejor modelo detras de una API para clasificacion en tiempo real.